In [1]:
# =============================================================================
# PROJECT       : Banking Stress Testing Framework
# VERSION       : Python V2 Migration
# MODULE        : 05 - Scenario Application
# NOTEBOOK      : 05_Scenario_Application.ipynb
# AUTHOR        : Umesh Zure
#
# PURPOSE:
# Apply every macroeconomic scenario to every segmented portfolio record using
# a Cartesian cross join that exactly reproduces the SAS V1 PROC SQL logic.
#
# INPUTS:
# output/segmented_portfolio.csv
# output/scenario_master.csv
#
# OUTPUTS:
# output/portfolio_scenario.csv
# validation/05_scenario_application_summary.csv
# validation/05_scenario_application_validation.csv
# validation/05_scenario_distribution.csv
# validation/05_portfolio_scenario_sample.csv
# logs/05_Scenario_Application.log
#
# SAS V1 PROCESS:
# 10,000 portfolio records
# x
# 3 scenarios
# =
# 30,000 portfolio-scenario records
#
# IMPORTANT:
# This module only expands the portfolio across scenarios.
# PD, LGD, EAD, and ECL stress calculations occur in later modules.
# =============================================================================

from datetime import datetime

module_start_timestamp = datetime.now()

print("=" * 80)
print("BANKING STRESS TESTING FRAMEWORK")
print("PYTHON V2 MIGRATION")
print("MODULE : 05_SCENARIO_APPLICATION")
print(
    "EXECUTION STARTED :",
    module_start_timestamp.strftime(
        "%Y-%m-%d %H:%M:%S"
    )
)
print("=" * 80)

BANKING STRESS TESTING FRAMEWORK
PYTHON V2 MIGRATION
MODULE : 05_SCENARIO_APPLICATION
EXECUTION STARTED : 2026-10-13 13:00:40


In [2]:
# =============================================================================
# BLOCK 1: ENVIRONMENT, PATHS, AND LOGGING
# =============================================================================

block_start_time = datetime.now()
block_status = "STARTED"
input_record_count = 0
output_record_count = 0
records_processed = 0

print("=" * 80)
print("MODULE 05: SCENARIO APPLICATION")
print("BLOCK 1 : ENVIRONMENT, PATHS, AND LOGGING")
print(
    "START TIME :",
    block_start_time.strftime("%Y-%m-%d %H:%M:%S")
)
print("=" * 80)

try:
    from google.colab import drive
    from pathlib import Path
    import logging
    import sys

    import numpy as np
    import pandas as pd

    drive.mount("/content/drive")

    PROJECT_ROOT = Path(
        "/content/drive/MyDrive/"
        "Banking_Stress_Testing_V2_Python"
    )

    PORTFOLIO_INPUT_FILE = (
        PROJECT_ROOT
        / "output"
        / "segmented_portfolio.csv"
    )

    SCENARIO_INPUT_FILE = (
        PROJECT_ROOT
        / "output"
        / "scenario_master.csv"
    )

    OUTPUT_FILE = (
        PROJECT_ROOT
        / "output"
        / "portfolio_scenario.csv"
    )

    VALIDATION_DIR = (
        PROJECT_ROOT
        / "validation"
    )

    LOG_DIR = (
        PROJECT_ROOT
        / "logs"
    )

    SUMMARY_FILE = (
        VALIDATION_DIR
        / "05_scenario_application_summary.csv"
    )

    VALIDATION_FILE = (
        VALIDATION_DIR
        / "05_scenario_application_validation.csv"
    )

    DISTRIBUTION_FILE = (
        VALIDATION_DIR
        / "05_scenario_distribution.csv"
    )

    SAMPLE_FILE = (
        VALIDATION_DIR
        / "05_portfolio_scenario_sample.csv"
    )

    LOG_FILE = (
        LOG_DIR
        / "05_Scenario_Application.log"
    )

    OUTPUT_FILE.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    VALIDATION_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    LOG_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    logger = logging.getLogger(
        "05_Scenario_Application"
    )

    logger.setLevel(logging.INFO)

    if logger.hasHandlers():
        logger.handlers.clear()

    log_formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s"
    )

    file_handler = logging.FileHandler(
        LOG_FILE,
        mode="w",
        encoding="utf-8"
    )

    file_handler.setFormatter(log_formatter)

    console_handler = logging.StreamHandler(
        sys.stdout
    )

    console_handler.setFormatter(log_formatter)

    logger.addHandler(file_handler)
    logger.addHandler(console_handler)

    if not PROJECT_ROOT.exists():
        raise FileNotFoundError(
            f"Project root was not found: {PROJECT_ROOT}"
        )

    if not PORTFOLIO_INPUT_FILE.exists():
        raise FileNotFoundError(
            "Module 03 segmented portfolio was not found: "
            f"{PORTFOLIO_INPUT_FILE}"
        )

    if not SCENARIO_INPUT_FILE.exists():
        raise FileNotFoundError(
            "Module 04 scenario master was not found: "
            f"{SCENARIO_INPUT_FILE}"
        )

    print(
        "Project Root Exists:",
        PROJECT_ROOT.exists()
    )

    print(
        "Module 03 Output Exists:",
        PORTFOLIO_INPUT_FILE.exists()
    )

    print(
        "Module 04 Output Exists:",
        SCENARIO_INPUT_FILE.exists()
    )

    print(
        "Portfolio Input:",
        PORTFOLIO_INPUT_FILE
    )

    print(
        "Scenario Input:",
        SCENARIO_INPUT_FILE
    )

    print(
        "Scenario-Applied Output:",
        OUTPUT_FILE
    )

    print(
        "Validation Directory:",
        VALIDATION_DIR
    )

    print(
        "Execution Log:",
        LOG_FILE
    )

    logger.info(
        "Environment, paths, and logging configured successfully."
    )

    block_status = "SUCCESS"

except Exception:
    block_status = "FAILED"

    if "logger" in locals():
        logger.exception(
            "Block 1 failed during environment configuration."
        )

    raise

finally:
    block_end_time = datetime.now()

    block_duration_seconds = round(
        (
            block_end_time
            - block_start_time
        ).total_seconds(),
        3
    )

    print("=" * 80)
    print("BLOCK STATUS       :", block_status)
    print("INPUT RECORDS      :", input_record_count)
    print("OUTPUT RECORDS     :", output_record_count)
    print("RECORDS PROCESSED  :", records_processed)
    print(
        "END TIME           :",
        block_end_time.strftime("%Y-%m-%d %H:%M:%S")
    )
    print(
        "DURATION (SECONDS) :",
        block_duration_seconds
    )
    print("=" * 80)

MODULE 05: SCENARIO APPLICATION
BLOCK 1 : ENVIRONMENT, PATHS, AND LOGGING
START TIME : 2026-10-13 13:03:47
Mounted at /content/drive
Project Root Exists: True
Module 03 Output Exists: True
Module 04 Output Exists: True
Portfolio Input: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/output/segmented_portfolio.csv
Scenario Input: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/output/scenario_master.csv
Scenario-Applied Output: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/output/portfolio_scenario.csv
Validation Directory: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/validation
Execution Log: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/logs/05_Scenario_Application.log
2026-10-13 13:04:16,840 | INFO | Environment, paths, and logging configured successfully.


INFO:05_Scenario_Application:Environment, paths, and logging configured successfully.


BLOCK STATUS       : SUCCESS
INPUT RECORDS      : 0
OUTPUT RECORDS     : 0
RECORDS PROCESSED  : 0
END TIME           : 2026-10-13 13:04:16
DURATION (SECONDS) : 29.588


In [3]:
# =============================================================================
# BLOCK 2: LOAD AND VALIDATE MODULE 03 AND MODULE 04 OUTPUTS
# =============================================================================

block_start_time = datetime.now()
block_status = "STARTED"
input_record_count = 0
output_record_count = 0
records_processed = 0

print("=" * 80)
print("MODULE 05: SCENARIO APPLICATION")
print("BLOCK 2 : LOAD AND VALIDATE INPUT DATASETS")
print(
    "START TIME :",
    block_start_time.strftime("%Y-%m-%d %H:%M:%S")
)
print("=" * 80)

try:
    required_portfolio_columns = [
        "Customer_ID",
        "Loan_ID",
        "Credit_Score",
        "Current_DPD",
        "Risk_Segment",
        "DPD_Bucket",
        "Outstanding_Balance",
    ]

    required_scenario_columns = [
        "Scenario_ID",
        "Scenario_Name",
        "GDP_Growth",
        "Unemployment_Rate",
        "Interest_Rate",
        "HPI_Change",
    ]

    segmented_portfolio = pd.read_csv(
        PORTFOLIO_INPUT_FILE,
        low_memory=False
    )

    scenario_master = pd.read_csv(
        SCENARIO_INPUT_FILE,
        low_memory=False
    )

    portfolio_record_count = len(
        segmented_portfolio
    )

    scenario_record_count = len(
        scenario_master
    )

    portfolio_column_count = len(
        segmented_portfolio.columns
    )

    scenario_column_count = len(
        scenario_master.columns
    )

    input_record_count = (
        portfolio_record_count
        + scenario_record_count
    )

    missing_portfolio_columns = sorted(
        set(required_portfolio_columns)
        - set(segmented_portfolio.columns)
    )

    missing_scenario_columns = sorted(
        set(required_scenario_columns)
        - set(scenario_master.columns)
    )

    duplicate_loan_ids = int(
        segmented_portfolio[
            "Loan_ID"
        ]
        .duplicated()
        .sum()
    )

    duplicate_scenario_ids = int(
        scenario_master[
            "Scenario_ID"
        ]
        .duplicated()
        .sum()
    )

    duplicate_scenario_names = int(
        scenario_master[
            "Scenario_Name"
        ]
        .duplicated()
        .sum()
    )

    if missing_portfolio_columns:
        raise ValueError(
            "Required portfolio columns are missing: "
            f"{missing_portfolio_columns}"
        )

    if missing_scenario_columns:
        raise ValueError(
            "Required scenario columns are missing: "
            f"{missing_scenario_columns}"
        )

    if duplicate_loan_ids > 0:
        raise ValueError(
            "Duplicate Loan_ID values were found "
            "in the segmented portfolio."
        )

    if duplicate_scenario_ids > 0:
        raise ValueError(
            "Duplicate Scenario_ID values were found."
        )

    if duplicate_scenario_names > 0:
        raise ValueError(
            "Duplicate Scenario_Name values were found."
        )

    expected_output_record_count = (
        portfolio_record_count
        * scenario_record_count
    )

    records_processed = input_record_count

    print("=" * 80)
    print("INPUT DATASETS LOADED SUCCESSFULLY")
    print("=" * 80)

    print(
        "Portfolio Records:",
        portfolio_record_count
    )

    print(
        "Portfolio Columns:",
        portfolio_column_count
    )

    print(
        "Scenario Records:",
        scenario_record_count
    )

    print(
        "Scenario Columns:",
        scenario_column_count
    )

    print(
        "Expected Cross-Join Records:",
        expected_output_record_count
    )

    print(
        "Missing Portfolio Columns:",
        missing_portfolio_columns
    )

    print(
        "Missing Scenario Columns:",
        missing_scenario_columns
    )

    print(
        "Duplicate Loan IDs:",
        duplicate_loan_ids
    )

    print(
        "Duplicate Scenario IDs:",
        duplicate_scenario_ids
    )

    print(
        "Duplicate Scenario Names:",
        duplicate_scenario_names
    )

    print("Input Validation Status: PASS")

    print("\nPortfolio Sample:")

    display(
        segmented_portfolio[
            [
                "Customer_ID",
                "Loan_ID",
                "Risk_Segment",
                "DPD_Bucket",
                "Outstanding_Balance",
            ]
        ]
        .head(5)
    )

    print("\nScenario Master:")

    display(
        scenario_master
    )

    logger.info(
        "Input datasets loaded successfully: "
        "%s portfolio records and %s scenarios.",
        portfolio_record_count,
        scenario_record_count
    )

    block_status = "SUCCESS"

except Exception:
    block_status = "FAILED"

    logger.exception(
        "Block 2 failed while loading input datasets."
    )

    raise

finally:
    block_end_time = datetime.now()

    block_duration_seconds = round(
        (
            block_end_time
            - block_start_time
        ).total_seconds(),
        3
    )

    print("=" * 80)
    print("BLOCK STATUS       :", block_status)
    print("INPUT RECORDS      :", input_record_count)
    print("OUTPUT RECORDS     :", output_record_count)
    print("RECORDS PROCESSED  :", records_processed)
    print(
        "END TIME           :",
        block_end_time.strftime("%Y-%m-%d %H:%M:%S")
    )
    print(
        "DURATION (SECONDS) :",
        block_duration_seconds
    )
    print("=" * 80)

MODULE 05: SCENARIO APPLICATION
BLOCK 2 : LOAD AND VALIDATE INPUT DATASETS
START TIME : 2026-10-13 13:06:22
INPUT DATASETS LOADED SUCCESSFULLY
Portfolio Records: 10000
Portfolio Columns: 24
Scenario Records: 3
Scenario Columns: 6
Expected Cross-Join Records: 30000
Missing Portfolio Columns: []
Missing Scenario Columns: []
Duplicate Loan IDs: 0
Duplicate Scenario IDs: 0
Duplicate Scenario Names: 0
Input Validation Status: PASS

Portfolio Sample:


,Customer_ID,Loan_ID,Risk_Segment,DPD_Bucket,Outstanding_Balance
0,1000001,50000001,Low Risk,Current,1733362.19
1,1000002,50000002,Low Risk,Current,62244073.93
2,1000003,50000003,High Risk,Current,303259.15
3,1000004,50000004,Medium Risk,Current,993246.23
4,1000005,50000005,Low Risk,Current,1654985.49



Scenario Master:


,Scenario_ID,Scenario_Name,GDP_Growth,Unemployment_Rate,Interest_Rate,HPI_Change
0,1,Baseline,6.0,5.0,6.0,0.0
1,2,Mild_Stress,3.0,7.0,8.0,-10.0
2,3,Severe_Stress,-2.0,10.0,11.0,-25.0


2026-10-13 13:06:23,438 | INFO | Input datasets loaded successfully: 10000 portfolio records and 3 scenarios.


INFO:05_Scenario_Application:Input datasets loaded successfully: 10000 portfolio records and 3 scenarios.


BLOCK STATUS       : SUCCESS
INPUT RECORDS      : 10003
OUTPUT RECORDS     : 0
RECORDS PROCESSED  : 10003
END TIME           : 2026-10-13 13:06:23
DURATION (SECONDS) : 0.692


In [7]:
# =============================================================================
# BLOCK 3: APPLY SCENARIOS USING SAS V1 CROSS-JOIN LOGIC
# =============================================================================

block_start_time = datetime.now()
block_status = "STARTED"

input_record_count = (
    portfolio_record_count
    + scenario_record_count
)

output_record_count = 0
records_processed = 0

print("=" * 80)
print("MODULE 05: SCENARIO APPLICATION")
print("BLOCK 3 : APPLY SCENARIOS USING CROSS JOIN")
print(
    "START TIME :",
    block_start_time.strftime(
        "%Y-%m-%d %H:%M:%S"
    )
)
print("=" * 80)

try:
    # -------------------------------------------------------------------------
    # Use explicit names for the portfolio rate and scenario rate.
    # This prevents ambiguous _x and _y suffixes after the cross join.
    # -------------------------------------------------------------------------

    portfolio_for_join = (
        segmented_portfolio
        .rename(
            columns={
                "Interest_Rate": "Loan_Interest_Rate"
            }
        )
        .copy()
    )

    scenario_for_join = (
        scenario_master
        .rename(
            columns={
                "Interest_Rate": "Scenario_Interest_Rate"
            }
        )
        .copy()
    )

    scenario_fields = [
        "Scenario_ID",
        "Scenario_Name",
        "GDP_Growth",
        "Unemployment_Rate",
        "Scenario_Interest_Rate",
        "HPI_Change",
    ]

    # Pandas cross merge reproduces the SAS PROC SQL CROSS JOIN.
    portfolio_scenario = portfolio_for_join.merge(
        scenario_for_join[
            scenario_fields
        ],
        how="cross",
        validate="many_to_many"
    )

    output_record_count = len(
        portfolio_scenario
    )

    output_column_count = len(
        portfolio_scenario.columns
    )

    records_processed = output_record_count

    if output_record_count != expected_output_record_count:
        raise ValueError(
            "Cross-join output count does not match "
            f"the expected {expected_output_record_count} records."
        )

    # Confirm that ambiguous merge suffixes were not created.
    ambiguous_columns = [
        column
        for column in portfolio_scenario.columns
        if column.endswith("_x")
        or column.endswith("_y")
    ]

    if ambiguous_columns:
        raise ValueError(
            "Ambiguous merge columns were created: "
            f"{ambiguous_columns}"
        )

    required_output_columns = [
        "Customer_ID",
        "Loan_ID",
        "Risk_Segment",
        "DPD_Bucket",
        "Loan_Interest_Rate",
        "Scenario_ID",
        "Scenario_Name",
        "GDP_Growth",
        "Unemployment_Rate",
        "Scenario_Interest_Rate",
        "HPI_Change",
    ]

    missing_output_columns = sorted(
        set(required_output_columns)
        - set(portfolio_scenario.columns)
    )

    if missing_output_columns:
        raise ValueError(
            "Required scenario-application columns "
            f"are missing: {missing_output_columns}"
        )

    print("=" * 80)
    print("SCENARIO APPLICATION COMPLETED")
    print("=" * 80)

    print(
        "Portfolio Records:",
        portfolio_record_count
    )

    print(
        "Scenario Records:",
        scenario_record_count
    )

    print(
        "Expected Output Records:",
        expected_output_record_count
    )

    print(
        "Actual Output Records:",
        output_record_count
    )

    print(
        "Output Columns:",
        output_column_count
    )

    print(
        "Missing Required Output Columns:",
        missing_output_columns
    )

    print(
        "Ambiguous Merge Columns:",
        ambiguous_columns
    )

    print(
        "Cross-Join Record Reconciliation: PASS"
    )

    display(
        portfolio_scenario[
            required_output_columns
        ]
        .head(20)
    )

    logger.info(
        "Scenario cross join completed successfully: "
        "%s output records and %s columns.",
        output_record_count,
        output_column_count
    )

    block_status = "SUCCESS"

except Exception:
    block_status = "FAILED"

    logger.exception(
        "Block 3 failed during scenario cross join."
    )

    raise

finally:
    block_end_time = datetime.now()

    block_duration_seconds = round(
        (
            block_end_time
            - block_start_time
        ).total_seconds(),
        3
    )

    print("=" * 80)
    print("BLOCK STATUS       :", block_status)
    print("INPUT RECORDS      :", input_record_count)
    print("OUTPUT RECORDS     :", output_record_count)
    print("RECORDS PROCESSED  :", records_processed)
    print(
        "END TIME           :",
        block_end_time.strftime(
            "%Y-%m-%d %H:%M:%S"
        )
    )
    print(
        "DURATION (SECONDS) :",
        block_duration_seconds
    )
    print("=" * 80)

MODULE 05: SCENARIO APPLICATION
BLOCK 3 : APPLY SCENARIOS USING CROSS JOIN
START TIME : 2026-10-13 13:57:14
SCENARIO APPLICATION COMPLETED
Portfolio Records: 10000
Scenario Records: 3
Expected Output Records: 30000
Actual Output Records: 30000
Output Columns: 30
Missing Required Output Columns: []
Ambiguous Merge Columns: []
Cross-Join Record Reconciliation: PASS


,Customer_ID,Loan_ID,Risk_Segment,DPD_Bucket,Loan_Interest_Rate,Scenario_ID,Scenario_Name,GDP_Growth,Unemployment_Rate,Scenario_Interest_Rate,HPI_Change
0,1000001,50000001,Low Risk,Current,19.40,1,Baseline,6.0,5.0,6.0,0.0
1,1000001,50000001,Low Risk,Current,19.40,2,Mild_Stress,3.0,7.0,8.0,-10.0
2,1000001,50000001,Low Risk,Current,19.40,3,Severe_Stress,-2.0,10.0,11.0,-25.0
3,1000002,50000002,Low Risk,Current,10.82,1,Baseline,6.0,5.0,6.0,0.0
4,1000002,50000002,Low Risk,Current,10.82,2,Mild_Stress,3.0,7.0,8.0,-10.0
5,1000002,50000002,Low Risk,Current,10.82,3,Severe_Stress,-2.0,10.0,11.0,-25.0
6,1000003,50000003,High Risk,Current,25.70,1,Baseline,6.0,5.0,6.0,0.0
7,1000003,50000003,High Risk,Current,25.70,2,Mild_Stress,3.0,7.0,8.0,-10.0
8,1000003,50000003,High Risk,Current,25.70,3,Severe_Stress,-2.0,10.0,11.0,-25.0
9,1000004,50000004,Medium Risk,Current,13.14,1,Baseline,6.0,5.0,6.0,0.0


2026-10-13 13:57:14,393 | INFO | Scenario cross join completed successfully: 30000 output records and 30 columns.


INFO:05_Scenario_Application:Scenario cross join completed successfully: 30000 output records and 30 columns.


BLOCK STATUS       : SUCCESS
INPUT RECORDS      : 10003
OUTPUT RECORDS     : 30000
RECORDS PROCESSED  : 30000
END TIME           : 2026-10-13 13:57:14
DURATION (SECONDS) : 0.042


In [6]:
print(portfolio_scenario.columns.tolist())

['Customer_ID', 'Loan_ID', 'Customer_Segment', 'Product_Type', 'Outstanding_Balance', 'Credit_Score', 'Previous_Credit_Score', 'Current_DPD', 'Previous_DPD', 'Interest_Rate_x', 'Loan_Term_Months', 'Remaining_Term_Months', 'Collateral_Value', 'Region', 'Reporting_Date', 'Original_Credit_Score', 'Credit_Score_Status', 'Credit_Score_Imputed_Flag', 'Original_Current_DPD', 'DPD_Status', 'Current_DPD_Imputed_Flag', 'Secured_Collateral_Missing_Flag', 'Risk_Segment', 'DPD_Bucket', 'Scenario_ID', 'Scenario_Name', 'GDP_Growth', 'Unemployment_Rate', 'Interest_Rate_y', 'HPI_Change']


In [8]:
# =============================================================================
# BLOCK 4: VALIDATE SCENARIO DISTRIBUTIONS
# =============================================================================

block_start_time = datetime.now()
block_status = "STARTED"
input_record_count = len(
    portfolio_scenario
)
output_record_count = 0
records_processed = 0

print("=" * 80)
print("MODULE 05: SCENARIO APPLICATION")
print("BLOCK 4 : VALIDATE SCENARIO DISTRIBUTIONS")
print(
    "START TIME :",
    block_start_time.strftime("%Y-%m-%d %H:%M:%S")
)
print("=" * 80)

try:
    expected_scenario_names = [
        "Baseline",
        "Mild_Stress",
        "Severe_Stress",
    ]

    scenario_distribution = (
        portfolio_scenario[
            "Scenario_Name"
        ]
        .value_counts(
            dropna=False
        )
        .reindex(
            expected_scenario_names,
            fill_value=0
        )
        .rename_axis(
            "Scenario_Name"
        )
        .reset_index(
            name="Record_Count"
        )
    )

    scenario_distribution[
        "Expected_Count"
    ] = portfolio_record_count

    scenario_distribution[
        "Portfolio_Percentage"
    ] = (
        scenario_distribution[
            "Record_Count"
        ]
        / output_record_count
        * 100
    ).round(2)

    scenario_distribution[
        "Status"
    ] = np.where(
        scenario_distribution[
            "Record_Count"
        ]
        ==
        scenario_distribution[
            "Expected_Count"
        ],
        "PASS",
        "FAIL"
    )

    failed_scenario_counts = (
        scenario_distribution.loc[
            scenario_distribution[
                "Status"
            ] != "PASS"
        ]
    )

    if not failed_scenario_counts.empty:
        raise ValueError(
            "One or more scenario record counts "
            "do not equal the portfolio count."
        )

    scenario_distribution_total = int(
        scenario_distribution[
            "Record_Count"
        ].sum()
    )

    if (
        scenario_distribution_total
        != expected_output_record_count
    ):
        raise ValueError(
            "Scenario distribution total does not "
            "reconcile to the expected output count."
        )

    output_record_count = len(
        scenario_distribution
    )

    records_processed = input_record_count

    print("=" * 80)
    print("SCENARIO DISTRIBUTION VALIDATION")
    print("=" * 80)

    display(
        scenario_distribution
    )

    print(
        "Scenario Distribution Total:",
        scenario_distribution_total
    )

    print(
        "Expected Distribution Total:",
        expected_output_record_count
    )

    print(
        "Scenario Distribution Reconciliation: PASS"
    )

    logger.info(
        "Scenario distribution validation passed."
    )

    block_status = "SUCCESS"

except Exception:
    block_status = "FAILED"

    logger.exception(
        "Block 4 failed during scenario distribution validation."
    )

    raise

finally:
    block_end_time = datetime.now()

    block_duration_seconds = round(
        (
            block_end_time
            - block_start_time
        ).total_seconds(),
        3
    )

    print("=" * 80)
    print("BLOCK STATUS       :", block_status)
    print("INPUT RECORDS      :", input_record_count)
    print("OUTPUT RECORDS     :", output_record_count)
    print("RECORDS PROCESSED  :", records_processed)
    print(
        "END TIME           :",
        block_end_time.strftime("%Y-%m-%d %H:%M:%S")
    )
    print(
        "DURATION (SECONDS) :",
        block_duration_seconds
    )
    print("=" * 80)

MODULE 05: SCENARIO APPLICATION
BLOCK 4 : VALIDATE SCENARIO DISTRIBUTIONS
START TIME : 2026-10-13 14:02:52
SCENARIO DISTRIBUTION VALIDATION


,Scenario_Name,Record_Count,Expected_Count,Portfolio_Percentage,Status
0,Baseline,10000,10000,inf,PASS
1,Mild_Stress,10000,10000,inf,PASS
2,Severe_Stress,10000,10000,inf,PASS


Scenario Distribution Total: 30000
Expected Distribution Total: 30000
Scenario Distribution Reconciliation: PASS
2026-10-13 14:02:52,818 | INFO | Scenario distribution validation passed.


INFO:05_Scenario_Application:Scenario distribution validation passed.


BLOCK STATUS       : SUCCESS
INPUT RECORDS      : 30000
OUTPUT RECORDS     : 3
RECORDS PROCESSED  : 30000
END TIME           : 2026-10-13 14:02:52
DURATION (SECONDS) : 0.02


In [9]:
# =============================================================================
# BLOCK 5: VALIDATE PORTFOLIO-SCENARIO UNIQUENESS AND COMPLETENESS
# =============================================================================

block_start_time = datetime.now()
block_status = "STARTED"
input_record_count = len(
    portfolio_scenario
)
output_record_count = input_record_count
records_processed = 0

print("=" * 80)
print("MODULE 05: SCENARIO APPLICATION")
print("BLOCK 5 : VALIDATE UNIQUENESS AND COMPLETENESS")
print(
    "START TIME :",
    block_start_time.strftime("%Y-%m-%d %H:%M:%S")
)
print("=" * 80)

try:
    duplicate_loan_scenario_keys = int(
        portfolio_scenario[
            [
                "Loan_ID",
                "Scenario_ID",
            ]
        ]
        .duplicated()
        .sum()
    )

    missing_scenario_ids = int(
        portfolio_scenario[
            "Scenario_ID"
        ]
        .isna()
        .sum()
    )

    missing_scenario_names = int(
        portfolio_scenario[
            "Scenario_Name"
        ]
        .isna()
        .sum()
    )

    missing_macro_values = int(
        portfolio_scenario[
            [
                "GDP_Growth",
                "Unemployment_Rate",
                "Scenario_Interest_Rate",
                "HPI_Change",
            ]
        ]
        .isna()
        .sum()
        .sum()
    )

    scenario_count_per_loan = (
        portfolio_scenario
        .groupby(
            "Loan_ID"
        )[
            "Scenario_ID"
        ]
        .nunique()
    )

    loans_with_incorrect_scenario_count = int(
        (
            scenario_count_per_loan
            != scenario_record_count
        )
        .sum()
    )

    unique_loan_scenario_keys = int(
        portfolio_scenario[
            [
                "Loan_ID",
                "Scenario_ID",
            ]
        ]
        .drop_duplicates()
        .shape[0]
    )

    if duplicate_loan_scenario_keys > 0:
        raise ValueError(
            "Duplicate Loan_ID and Scenario_ID "
            "combinations were found."
        )

    if missing_scenario_ids > 0:
        raise ValueError(
            "Missing Scenario_ID values were found."
        )

    if missing_scenario_names > 0:
        raise ValueError(
            "Missing Scenario_Name values were found."
        )

    if missing_macro_values > 0:
        raise ValueError(
            "Missing macroeconomic values were found."
        )

    if loans_with_incorrect_scenario_count > 0:
        raise ValueError(
            "One or more loans do not have exactly "
            f"{scenario_record_count} scenarios."
        )

    records_processed = input_record_count

    print("=" * 80)
    print("PORTFOLIO-SCENARIO COMPLETENESS VALIDATION")
    print("=" * 80)

    print(
        "Unique Loan-Scenario Keys:",
        unique_loan_scenario_keys
    )

    print(
        "Duplicate Loan-Scenario Keys:",
        duplicate_loan_scenario_keys
    )

    print(
        "Missing Scenario IDs:",
        missing_scenario_ids
    )

    print(
        "Missing Scenario Names:",
        missing_scenario_names
    )

    print(
        "Missing Macroeconomic Values:",
        missing_macro_values
    )

    print(
        "Loans with Incorrect Scenario Count:",
        loans_with_incorrect_scenario_count
    )

    print(
        "Expected Scenarios Per Loan:",
        scenario_record_count
    )

    print(
        "Portfolio-Scenario Completeness Status: PASS"
    )

    print("\nScenario Count Per Loan Summary:")

    display(
        scenario_count_per_loan
        .describe()
        .to_frame(
            name="Scenario_Count_Per_Loan"
        )
    )

    logger.info(
        "Portfolio-scenario uniqueness and "
        "completeness validation passed."
    )

    block_status = "SUCCESS"

except Exception:
    block_status = "FAILED"

    logger.exception(
        "Block 5 failed during completeness validation."
    )

    raise

finally:
    block_end_time = datetime.now()

    block_duration_seconds = round(
        (
            block_end_time
            - block_start_time
        ).total_seconds(),
        3
    )

    print("=" * 80)
    print("BLOCK STATUS       :", block_status)
    print("INPUT RECORDS      :", input_record_count)
    print("OUTPUT RECORDS     :", output_record_count)
    print("RECORDS PROCESSED  :", records_processed)
    print(
        "END TIME           :",
        block_end_time.strftime("%Y-%m-%d %H:%M:%S")
    )
    print(
        "DURATION (SECONDS) :",
        block_duration_seconds
    )
    print("=" * 80)

MODULE 05: SCENARIO APPLICATION
BLOCK 5 : VALIDATE UNIQUENESS AND COMPLETENESS
START TIME : 2026-10-13 14:08:34
PORTFOLIO-SCENARIO COMPLETENESS VALIDATION
Unique Loan-Scenario Keys: 30000
Duplicate Loan-Scenario Keys: 0
Missing Scenario IDs: 0
Missing Scenario Names: 0
Missing Macroeconomic Values: 0
Loans with Incorrect Scenario Count: 0
Expected Scenarios Per Loan: 3
Portfolio-Scenario Completeness Status: PASS

Scenario Count Per Loan Summary:


,Scenario_Count_Per_Loan
count,10000.0
mean,3.0
std,0.0
min,3.0
25%,3.0
50%,3.0
75%,3.0
max,3.0


2026-10-13 14:08:34,833 | INFO | Portfolio-scenario uniqueness and completeness validation passed.


INFO:05_Scenario_Application:Portfolio-scenario uniqueness and completeness validation passed.


BLOCK STATUS       : SUCCESS
INPUT RECORDS      : 30000
OUTPUT RECORDS     : 30000
RECORDS PROCESSED  : 30000
END TIME           : 2026-10-13 14:08:34
DURATION (SECONDS) : 0.021


In [10]:
# =============================================================================
# BLOCK 6: FINAL VALIDATION AND MODULE SUMMARY
# =============================================================================

block_start_time = datetime.now()
block_status = "STARTED"
input_record_count = (
    portfolio_record_count
    + scenario_record_count
)
output_record_count = len(
    portfolio_scenario
)
records_processed = 0

print("=" * 80)
print("MODULE 05: SCENARIO APPLICATION")
print("BLOCK 6 : FINAL VALIDATION AND MODULE SUMMARY")
print(
    "START TIME :",
    block_start_time.strftime("%Y-%m-%d %H:%M:%S")
)
print("=" * 80)

try:
    validation_results = pd.DataFrame({
        "Validation_Check": [
            "Portfolio Record Count",
            "Scenario Record Count",
            "Expected Cross-Join Count",
            "Actual Cross-Join Count",
            "Scenario Distribution Total",
            "Duplicate Loan-Scenario Keys",
            "Missing Scenario IDs",
            "Missing Scenario Names",
            "Missing Macroeconomic Values",
            "Loans with Incorrect Scenario Count",
        ],
        "Expected_Result": [
            10000,
            3,
            expected_output_record_count,
            expected_output_record_count,
            expected_output_record_count,
            0,
            0,
            0,
            0,
            0,
        ],
        "Actual_Result": [
            portfolio_record_count,
            scenario_record_count,
            expected_output_record_count,
            output_record_count,
            scenario_distribution_total,
            duplicate_loan_scenario_keys,
            missing_scenario_ids,
            missing_scenario_names,
            missing_macro_values,
            loans_with_incorrect_scenario_count,
        ],
    })

    validation_results[
        "Status"
    ] = np.where(
        validation_results[
            "Expected_Result"
        ].astype(str)
        ==
        validation_results[
            "Actual_Result"
        ].astype(str),
        "PASS",
        "FAIL"
    )

    scenario_application_summary = pd.DataFrame({
        "Metric": [
            "Portfolio Input Records",
            "Scenario Input Records",
            "Portfolio Input Columns",
            "Scenario Input Columns",
            "Scenario-Applied Output Records",
            "Scenario-Applied Output Columns",
            "Scenarios Per Loan",
            "Baseline Records",
            "Mild Stress Records",
            "Severe Stress Records",
            "Processing Timestamp",
        ],
        "Result": [
            portfolio_record_count,
            scenario_record_count,
            portfolio_column_count,
            scenario_column_count,
            output_record_count,
            output_column_count,
            scenario_record_count,
            int(
                (
                    portfolio_scenario[
                        "Scenario_Name"
                    ] == "Baseline"
                ).sum()
            ),
            int(
                (
                    portfolio_scenario[
                        "Scenario_Name"
                    ] == "Mild_Stress"
                ).sum()
            ),
            int(
                (
                    portfolio_scenario[
                        "Scenario_Name"
                    ] == "Severe_Stress"
                ).sum()
            ),
            datetime.now(),
        ],
    })

    failed_validations = validation_results.loc[
        validation_results[
            "Status"
        ] != "PASS"
    ]

    if not failed_validations.empty:
        raise ValueError(
            "One or more Module 05 validation "
            "checks failed."
        )

    records_processed = output_record_count

    print("=" * 80)
    print("FINAL SCENARIO APPLICATION VALIDATION")
    print("=" * 80)

    display(
        validation_results
    )

    print("\nScenario Application Summary:")

    display(
        scenario_application_summary
    )

    print(
        "Overall Module 05 Validation Status: PASS"
    )

    logger.info(
        "All Module 05 validation checks passed."
    )

    block_status = "SUCCESS"

except Exception:
    block_status = "FAILED"

    logger.exception(
        "Block 6 failed during final validation."
    )

    raise

finally:
    block_end_time = datetime.now()

    block_duration_seconds = round(
        (
            block_end_time
            - block_start_time
        ).total_seconds(),
        3
    )

    print("=" * 80)
    print("BLOCK STATUS       :", block_status)
    print("INPUT RECORDS      :", input_record_count)
    print("OUTPUT RECORDS     :", output_record_count)
    print("RECORDS PROCESSED  :", records_processed)
    print(
        "END TIME           :",
        block_end_time.strftime("%Y-%m-%d %H:%M:%S")
    )
    print(
        "DURATION (SECONDS) :",
        block_duration_seconds
    )
    print("=" * 80)

MODULE 05: SCENARIO APPLICATION
BLOCK 6 : FINAL VALIDATION AND MODULE SUMMARY
START TIME : 2026-10-13 14:09:47
FINAL SCENARIO APPLICATION VALIDATION


,Validation_Check,Expected_Result,Actual_Result,Status
0,Portfolio Record Count,10000,10000,PASS
1,Scenario Record Count,3,3,PASS
2,Expected Cross-Join Count,30000,30000,PASS
3,Actual Cross-Join Count,30000,30000,PASS
4,Scenario Distribution Total,30000,30000,PASS
5,Duplicate Loan-Scenario Keys,0,0,PASS
6,Missing Scenario IDs,0,0,PASS
7,Missing Scenario Names,0,0,PASS
8,Missing Macroeconomic Values,0,0,PASS
9,Loans with Incorrect Scenario Count,0,0,PASS



Scenario Application Summary:


,Metric,Result
0,Portfolio Input Records,10000
1,Scenario Input Records,3
2,Portfolio Input Columns,24
3,Scenario Input Columns,6
4,Scenario-Applied Output Records,30000
5,Scenario-Applied Output Columns,30
6,Scenarios Per Loan,3
7,Baseline Records,10000
8,Mild Stress Records,10000
9,Severe Stress Records,10000


Overall Module 05 Validation Status: PASS
2026-10-13 14:09:47,250 | INFO | All Module 05 validation checks passed.


INFO:05_Scenario_Application:All Module 05 validation checks passed.


BLOCK STATUS       : SUCCESS
INPUT RECORDS      : 10003
OUTPUT RECORDS     : 30000
RECORDS PROCESSED  : 30000
END TIME           : 2026-10-13 14:09:47
DURATION (SECONDS) : 0.036


In [11]:
# =============================================================================
# BLOCK 7: SAVE OUTPUTS, VERIFY FILES, AND CLOSE MODULE
# =============================================================================

block_start_time = datetime.now()
block_status = "STARTED"
input_record_count = len(
    portfolio_scenario
)
output_record_count = 0
records_processed = 0

print("=" * 80)
print("MODULE 05: SCENARIO APPLICATION")
print("BLOCK 7 : SAVE OUTPUTS AND VERIFY FILES")
print(
    "START TIME :",
    block_start_time.strftime("%Y-%m-%d %H:%M:%S")
)
print("=" * 80)

try:
    portfolio_scenario.to_csv(
        OUTPUT_FILE,
        index=False,
        date_format="%Y-%m-%d"
    )

    scenario_application_summary.to_csv(
        SUMMARY_FILE,
        index=False
    )

    validation_results.to_csv(
        VALIDATION_FILE,
        index=False
    )

    scenario_distribution.to_csv(
        DISTRIBUTION_FILE,
        index=False
    )

    portfolio_scenario.head(20).to_csv(
        SAMPLE_FILE,
        index=False,
        date_format="%Y-%m-%d"
    )

    logger.info(
        "Scenario-applied portfolio and validation "
        "evidence saved successfully."
    )

    expected_output_files = [
        OUTPUT_FILE,
        SUMMARY_FILE,
        VALIDATION_FILE,
        DISTRIBUTION_FILE,
        SAMPLE_FILE,
        LOG_FILE,
    ]

    output_verification = pd.DataFrame({
        "File_Name": [
            file_path.name
            for file_path
            in expected_output_files
        ],
        "Exists": [
            file_path.exists()
            for file_path
            in expected_output_files
        ],
        "Size_Bytes": [
            (
                file_path.stat().st_size
                if file_path.exists()
                else 0
            )
            for file_path
            in expected_output_files
        ],
        "Full_Path": [
            str(file_path)
            for file_path
            in expected_output_files
        ],
    })

    if not output_verification[
        "Exists"
    ].all():
        raise FileNotFoundError(
            "One or more Module 05 output files "
            "were not created."
        )

    saved_output = pd.read_csv(
        OUTPUT_FILE,
        low_memory=False
    )

    output_record_count = len(
        saved_output
    )

    if output_record_count != expected_output_record_count:
        raise ValueError(
            "Saved scenario-applied output does not "
            "contain the expected 30,000 records."
        )

    records_processed = output_record_count

    block_status = "SUCCESS"

except Exception:
    block_status = "FAILED"

    logger.exception(
        "Block 7 failed while saving or verifying outputs."
    )

    raise

finally:
    block_end_time = datetime.now()

    block_duration_seconds = round(
        (
            block_end_time
            - block_start_time
        ).total_seconds(),
        3
    )

    print("=" * 80)
    print("BLOCK STATUS       :", block_status)
    print("INPUT RECORDS      :", input_record_count)
    print("OUTPUT RECORDS     :", output_record_count)
    print("RECORDS PROCESSED  :", records_processed)
    print(
        "END TIME           :",
        block_end_time.strftime("%Y-%m-%d %H:%M:%S")
    )
    print(
        "DURATION (SECONDS) :",
        block_duration_seconds
    )
    print("=" * 80)

module_end_timestamp = datetime.now()

module_duration_seconds = round(
    (
        module_end_timestamp
        - module_start_timestamp
    ).total_seconds(),
    3
)

print("=" * 80)
print("PYTHON V2 - MODULE 05 COMPLETED")
print("SCENARIO APPLICATION EXECUTED SUCCESSFULLY")
print("=" * 80)

print(
    "Portfolio Input Records:",
    portfolio_record_count
)

print(
    "Scenario Input Records:",
    scenario_record_count
)

print(
    "Scenario-Applied Records:",
    output_record_count
)

print(
    "Scenario-Applied Output:",
    OUTPUT_FILE
)

print(
    "Validation Summary:",
    SUMMARY_FILE
)

print(
    "Validation Evidence:",
    VALIDATION_FILE
)

print(
    "Scenario Distribution:",
    DISTRIBUTION_FILE
)

print(
    "Output Sample:",
    SAMPLE_FILE
)

print(
    "Execution Log:",
    LOG_FILE
)

print("\nOutput File Verification:")

display(
    output_verification
)

print("=" * 80)
print("MODULE STATUS             : SUCCESS")
print(
    "MODULE START              :",
    module_start_timestamp.strftime(
        "%Y-%m-%d %H:%M:%S"
    )
)
print(
    "MODULE END                :",
    module_end_timestamp.strftime(
        "%Y-%m-%d %H:%M:%S"
    )
)
print(
    "TOTAL DURATION (SECONDS)  :",
    module_duration_seconds
)
print(
    "TOTAL RECORDS PROCESSED   :",
    output_record_count
)
print("=" * 80)

print(
    "All Module 05 output files verified successfully."
)

logger.info(
    "Module 05 completed successfully in %s seconds.",
    module_duration_seconds
)

MODULE 05: SCENARIO APPLICATION
BLOCK 7 : SAVE OUTPUTS AND VERIFY FILES
START TIME : 2026-10-13 14:12:44
2026-10-13 14:12:45,424 | INFO | Scenario-applied portfolio and validation evidence saved successfully.


INFO:05_Scenario_Application:Scenario-applied portfolio and validation evidence saved successfully.


BLOCK STATUS       : SUCCESS
INPUT RECORDS      : 30000
OUTPUT RECORDS     : 30000
RECORDS PROCESSED  : 30000
END TIME           : 2026-10-13 14:12:45
DURATION (SECONDS) : 0.653
PYTHON V2 - MODULE 05 COMPLETED
SCENARIO APPLICATION EXECUTED SUCCESSFULLY
Portfolio Input Records: 10000
Scenario Input Records: 3
Scenario-Applied Records: 30000
Scenario-Applied Output: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/output/portfolio_scenario.csv
Validation Summary: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/validation/05_scenario_application_summary.csv
Validation Evidence: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/validation/05_scenario_application_validation.csv
Scenario Distribution: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/validation/05_scenario_distribution.csv
Output Sample: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/validation/05_portfolio_scenario_sample.csv
Execution Log: /content/drive/MyDrive/Banking_Stress_Testing_V2

,File_Name,Exists,Size_Bytes,Full_Path
0,portfolio_scenario.csv,True,6253454,/content/drive/MyDrive/Banking_Stress_Testing_...
1,05_scenario_application_summary.csv,True,340,/content/drive/MyDrive/Banking_Stress_Testing_...
2,05_scenario_application_validation.csv,True,437,/content/drive/MyDrive/Banking_Stress_Testing_...
3,05_scenario_distribution.csv,True,168,/content/drive/MyDrive/Banking_Stress_Testing_...
4,05_portfolio_scenario_sample.csv,True,4609,/content/drive/MyDrive/Banking_Stress_Testing_...
5,05_Scenario_Application.log,True,1617,/content/drive/MyDrive/Banking_Stress_Testing_...


MODULE STATUS             : SUCCESS
MODULE START              : 2026-10-13 13:00:40
MODULE END                : 2026-10-13 14:12:45
TOTAL DURATION (SECONDS)  : 4325.286
TOTAL RECORDS PROCESSED   : 30000
All Module 05 output files verified successfully.
2026-10-13 14:12:45,509 | INFO | Module 05 completed successfully in 4325.286 seconds.


INFO:05_Scenario_Application:Module 05 completed successfully in 4325.286 seconds.
